# 🌳 Color Feature Comparison (Đánh giá đặc trưng Màu Sắc)
Notebook này tập trung vào việc **so sánh từng đặc trưng thuộc nhóm màu sắc** giữa các cặp ảnh cái cây. Mục tiêu là phân tích và kiểm tra **điểm mạnh, điểm yếu** của từng thuộc tính thông qua đối chiếu định lượng (vector) và trực quan hóa (visualizations).

Các đặc trưng đang sử dụng trong `features/color.py`:
- `mean_h` (Trung bình màu) & `mean_s` (Trung bình độ bão hòa)
- `leaf_var` (Độ biến thiên xanh lá)
- `bg_ratio` (Tỉ lệ màu nâu/xám - diện tích cành/thân/lá úa)
- `hue_hist` (5 nhóm màu cơ bản của cái cây)

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Setup đường dẫn tương đối (sys.path) để gọi thẳng thư mục src
notebook_dir = Path(".").resolve()
root_dir = notebook_dir.parent
sys.path.append(str(root_dir / "src"))

from features.color import extract_color

# Tên các thuộc tính (9 đặc trưng màu sắc như thiết kế trong extractor.py)
COLOR_FEATURE_NAMES = [
    "Mean Hue (Màu)", "Mean Sat (Tương phản)", "Leaf Color Var (Biến thiên lá)",
    "Brown/Gray Ratio (Tỷ lệ Thân/Úa)", 
    "Bin 1 (Đỏ/Cam)", "Bin 2 (Vàng)", "Bin 3 (Xanh lá)", "Bin 4 (Xanh lam)", "Bin 5 (Hồng/Tím)"
]

print(f"[READY] Import thành công các module cần thiết.")

In [ ]:
def extract_and_visualize_comparison(path1, path2):
    """
    Hàm nhận 2 đường dẫn ảnh cây (Ảnh A & Ảnh B), tiến hành so sánh chuyên sâu nhóm đặc trưng màu sắc.
    Nó sẽ visualize: 
    1. Hình ảnh gốc bọc lại bằng Mask.
    2. Biểu đồ Radar/Bar đặc trưng 5 nhóm HueBins & Các chỉ số khác.
    3. Bảng so sánh % Chênh lệch (Difference) giúp nhìn ra điểm yếu/mạnh.
    """
    # 1. Đọc 2 ảnh
    img1 = cv2.imread(str(path1))
    img2 = cv2.imread(str(path2))

    if img1 is None or img2 is None:
        print("[FAIL] Mất file ảnh truyền vào.")
        return
        
    img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
    img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
    
    # 2. Xử lý qua hệ thống rút trích màu
    v1 = np.array(extract_color(img1))
    v2 = np.array(extract_color(img2))
    
    # Tính độ lệch (Absolute Difference)
    diff = np.abs(v1 - v2)
    diff_percent = np.where(np.maximum(v1, v2) > 0, (diff / np.maximum(v1, v2)) * 100, 0)
    
    # Tạo DataFrame để xuất báo cáo
    df = pd.DataFrame({
        "Attribute": COLOR_FEATURE_NAMES,
        "Tree A": np.round(v1, 3),
        "Tree B": np.round(v2, 3),
        "Abs Diff": np.round(diff, 3),
        "Max Diff %": [f"{p:.1f}%" for p in diff_percent]
    })
    
    # 3. TRỰC QUAN HOÁ 
    plt.figure(figsize=(18, 10))
    
    # Ảnh A 
    plt.subplot(2, 3, 1)
    plt.title(f"Tree A: {Path(path1).name}")
    plt.imshow(img1_rgb)
    plt.axis('off')
    
    # Ảnh B
    plt.subplot(2, 3, 3)
    plt.title(f"Tree B: {Path(path2).name}")
    plt.imshow(img2_rgb)
    plt.axis('off')
    
    # Bar Chart so sánh 4 thuộc tính cơ bản (Primary, Variance, Branch)
    plt.subplot(2, 3, 4)
    x = np.arange(4)
    width = 0.35
    plt.bar(x - width/2, v1[:4], width, label='Tree A', color='limegreen')
    plt.bar(x + width/2, v2[:4], width, label='Tree B', color='orange')
    plt.xticks(x, ["Mean Hue", "Mean Sat", "Leaf Vari.", "Branch/Gray Ratio"], rotation=15)
    plt.title("So sánh 4 thuộc tính Cơ Bản")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    # Bar Chart so sánh 5 Histogram Bins
    plt.subplot(2, 3, 6)
    x_hist = np.arange(5)
    plt.bar(x_hist - width/2, v1[4:], width, label='Tree A', color='limegreen')
    plt.bar(x_hist + width/2, v2[4:], width, label='Tree B', color='orange')
    plt.xticks(x_hist, ["Bin 1 (Đỏ/Cam)", "Bin 2 (Vàng)", "Bin 3 (Xanh lục)", "Bin 4 (Xanh lam)", "Bin 5 (Hồng/Tím)"], rotation=15)
    plt.title("So sánh Phân bổ hệ màu sắc (5 Bins)")
    plt.ylim(0, 1.0)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # Hiển thị bảng số liệu chi tiết
    print(f"\n✨ BẢNG PHÂN TÍCH SO SÁNH GIỮA {Path(path1).name} VÀ {Path(path2).name} ✨")
    display(df.style.background_gradient(cmap='Reds', subset=['Abs Diff']))

print("[READY] Load hàm phân tích thành công.")

## Tình huống 1: Hai cây giống loài nhau nhưng khác ánh sáng/gốc chụp
Tìm điểm mạnh của `mean_h` (Trị trung bình màu) và `hue_hist` (Biểu đồ nhóm). Chúng sẽ **giống nhau** để nhận dạng loài. Đồng thời nhận diện điểm yếu/sự sai lệch ở `mean_s` (Tương phản ánh sáng bị nhoè).

In [ ]:
# Bạn có thể thay đường dẫn tới 2 file ảnh mẫu bất kì (cùng lá xanh)
img_path_A = root_dir / "data/raw/1.jpg"
img_path_B = root_dir / "data/raw/2.jpg"

extract_and_visualize_comparison(img_path_A, img_path_B)

## Tình huống 2: Hai cây khác loài, hoặc có màu lá khác biệt (cây thu vàng vs cây xanh)
Kiểm tra khả năng phân biệt mạnh mẽ của biểu đồ Hue histogram và thuộc tính Mean Hue. Trong khi Mean_H có thể kéo ngang lại, Histogram (`hue_hist`) lật tẩy hoàn toàn sự phân mảnh màu.

In [ ]:
img_path_C = root_dir / "data/raw/1.jpg"   # Lá xanh
img_path_D = root_dir / "data/raw/10.jpg"  # Nếu bạn có 1 ảnh lá đổi màu

extract_and_visualize_comparison(img_path_C, img_path_D)